# 01 — Feature engineering

**Week 2, Task 2 · Prepared for a Colab run.** Execute all cells in order and save the aggregate outputs. No real-data results have been recorded in this copy.

Build the same 33 leakage-safe feature columns for the published `train` and `validation` splits. The source code is pinned to a repository commit and the dataset to the Week 1 revision. The notebook requests only those two named splits; the `test` split stays closed. Stored labels and decoder predictions are not used to build `X` or in any check here. See the [task guide](../docs/WEEK_2_TASK_GUIDE.md) and [feature note](../results/tables/week2_feature_checks.md).

## 1. Install and load pinned project code

This Colab setup downloads only the reusable loader, feature extractor, and their configuration from the recorded repository commit. It uses a separate working folder, so it does not alter repository files. If feature code changes later, update `SOURCE_REF`, rerun from the start, and record the new commit in the findings note.

In [ ]:
%pip install -q --disable-pip-version-check datasets numpy pandas PyYAML

In [ ]:
import importlib.metadata
import json
import os
import sys
from pathlib import Path
from urllib.request import urlopen

REPO = "rifahnanjiba02-arch/selective-decoder-routing"
SOURCE_REF = "58b14a3afe79d214ab0c302c79ce80a843624a4b"  # Includes Week 2 Task 1 extractor.
ROOT = Path(os.environ.get("SDR_NOTEBOOK_ROOT", "/content/selective_decoder_week2"))
REQUIRED_FILES = (
    "configs/data.yaml",
    "configs/features.yaml",
    "src/router/__init__.py",
    "src/router/data.py",
    "src/router/features.py",
)

with urlopen(f"https://api.github.com/repos/{REPO}/commits/{SOURCE_REF}") as response:
    repository_commit = json.load(response)["sha"]
assert repository_commit == SOURCE_REF
for relative_path in REQUIRED_FILES:
    destination = ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    with urlopen(f"https://raw.githubusercontent.com/{REPO}/{repository_commit}/{relative_path}") as response:
        destination.write_bytes(response.read())
os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print(f"Loaded {len(REQUIRED_FILES)} project files at {repository_commit}")

## 2. Record run settings and the exact feature schema

The repository data config leaves the dataset revision unset. Write a pinned copy in the Colab working folder, then use it for both named split requests. The activity limit of 26 comes from the documented training schema (`rounds` up to 25, with `time_steps = rounds + 1`); validation data does not set this limit. Shorter sequences have trailing zero activity slots. Longer sequences raise an error.

In [ ]:
import yaml
from router.features import (
    ACTIVITY_COLUMNS, FEATURE_COLUMNS, MAX_TIME_STEPS, NUMERIC_COLUMNS,
    SOURCE_FIELDS, VALID_BASES, extract_features, row_features,
)

DATASET_REVISION = "9c51b27ffc51ed7ff3ef3a72167ebf4862d1f2f3"
with open(ROOT / "configs/data.yaml", encoding="utf-8") as file:
    data_config = yaml.safe_load(file)
with open(ROOT / "configs/features.yaml", encoding="utf-8") as file:
    feature_config = yaml.safe_load(file)
assert data_config["dataset_name"] == "Bauxitiego/surface-code-syndromes"
assert data_config.get("dataset_revision") in (None, DATASET_REVISION)
assert feature_config["max_time_steps"] == MAX_TIME_STEPS == 26
assert tuple(feature_config["source_fields"]) == SOURCE_FIELDS
assert feature_config["padding_mask"] == "round_widths"
expected_columns = tuple(
    column
    for candidate in feature_config["compact_features"]
    for column in (ACTIVITY_COLUMNS if candidate == "activity_across_rounds" else (candidate,))
)
assert expected_columns == FEATURE_COLUMNS
assert len(FEATURE_COLUMNS) == 33

pinned_config = dict(data_config)
pinned_config["dataset_revision"] = DATASET_REVISION
config_path = ROOT / "configs/data_colab_pinned.yaml"
config_path.write_text(yaml.safe_dump(pinned_config, sort_keys=False), encoding="utf-8")
SAMPLE_SIZES = {
    "train": pinned_config["development"]["train_size"],
    "validation": pinned_config["development"]["validation_size"],
}
SEED = pinned_config["sampling_seed"]
assert SAMPLE_SIZES == {"train": 50_000, "validation": 10_000}
assert SEED == 42

run_settings = {
    "repository_commit": repository_commit,
    "dataset": pinned_config["dataset_name"],
    "dataset_revision": DATASET_REVISION,
    "published_splits": tuple(SAMPLE_SIZES),
    "sample_sizes": SAMPLE_SIZES,
    "sampling_seed_each_split": SEED,
    "published_split_seed": pinned_config["source_split_seed"],
    "source_fields": SOURCE_FIELDS,
    "feature_columns": FEATURE_COLUMNS,
    "max_time_steps": MAX_TIME_STEPS,
    "package_versions": {
        name: importlib.metadata.version(name)
        for name in ("datasets", "numpy", "pandas", "PyYAML")
    },
}
for name, value in run_settings.items():
    print(f"{name}: {value}")

## 3. Load and sample the published train and validation splits

Call the existing loader once per named split and sample each independently with seed 42. No random train/validation repartition is made. The loader may download or prepare dataset files internally; the code below requests and selects only `train` and `validation` rows. Loading a full split before sampling may take time in Colab.

In [ ]:
from router.data import load_surface_code_dataset, sample_split

requested_splits = []
samples = {}
full_split_sizes = {}
for split_name in ("train", "validation"):
    requested_splits.append(split_name)
    loaded, loaded_config = load_surface_code_dataset(
        config_path=str(config_path), split_name=split_name
    )
    assert set(loaded) == {split_name}
    assert loaded_config["dataset_revision"] == DATASET_REVISION
    full_split_sizes[split_name] = len(loaded[split_name])
    samples[split_name] = sample_split(
        loaded, split_name=split_name,
        sample_size=SAMPLE_SIZES[split_name], seed=SEED,
    )
    assert len(samples[split_name]) == SAMPLE_SIZES[split_name]
assert requested_splits == ["train", "validation"]
assert "test" not in requested_splits
print("Requested splits:", requested_splits)
for split_name in requested_splits:
    print(f"{split_name}: {full_split_sizes[split_name]:,} published rows; "
          f"{len(samples[split_name]):,} sampled rows")

## 4. Extract the configured features

`extract_features()` reads only the seven allowed source fields. It returns named arrays in the configured order. The `basis` string stays categorical; there is no encoding, scaling, fitting, or target construction in this notebook.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

FORBIDDEN = {
    "label", "pymatching", "correlated_matching", "decoder_correct",
    "pymatching_correct", "correlated_matching_correct", "y_router", "experiment",
}
assert not (FORBIDDEN & set(SOURCE_FIELDS))
assert not (FORBIDDEN & set(FEATURE_COLUMNS))
X = {}
for split_name in requested_splits:
    X[split_name] = pd.DataFrame(extract_features(samples[split_name]))
    assert tuple(X[split_name].columns) == expected_columns
    assert X[split_name].shape == (SAMPLE_SIZES[split_name], len(expected_columns))
    assert not (FORBIDDEN & set(X[split_name].columns))
print("Feature columns, in order:")
print(", ".join(expected_columns))
print("Train shape:", X["train"].shape, "Validation shape:", X["validation"].shape)

## Feature definitions

| Feature | Definition |
| --- | --- |
| `detection_event_count` | Number of 1s in real syndrome positions. |
| `event_density` | Event count divided by `sum(round_widths)`; 0.0 when the denominator is zero. |
| `activity_round_00` … `activity_round_25` | Event count in each time step, including the final measurement step. Unused trailing slots are zero; more than 26 steps raises an error. |
| `active_rounds` | Number of time steps with at least one real event, including the final measurement step. |
| `maximum_round_activity` | Largest real event count in any time step. |
| `distance`, `rounds` | Permitted numeric experiment metadata. |
| `basis` | Permitted `X`/`Z` category, kept as a string for later train-only encoding. |

Only the first `round_widths[t]` positions of each flattened syndrome time step are real. All other positions are padding and contribute to no count or density.

## 5. Validate rows, values, padding, and leakage boundary

These checks cover every selected row. The independent padding check recomputes counts from each row's real positions and confirms that the feature table agrees. The synthetic check deliberately puts 1s in padding, because naturally padded values may all be zero.

In [ ]:
check_rows = []
for split_name in requested_splits:
    frame = X[split_name]
    numeric = frame.loc[:, NUMERIC_COLUMNS].to_numpy(dtype=float)
    assert tuple(frame.columns) == expected_columns
    assert len(frame) == SAMPLE_SIZES[split_name]
    assert np.isfinite(numeric).all()
    assert frame["basis"].notna().all()
    assert set(frame["basis"].unique()).issubset(set(VALID_BASES))
    assert frame["detection_event_count"].ge(0).all()
    assert frame["event_density"].between(0, 1).all()
    assert frame["active_rounds"].le(frame["rounds"] + 1).all()
    assert frame["maximum_round_activity"].le(frame["detection_event_count"]).all()
    assert (frame.loc[:, ACTIVITY_COLUMNS].sum(axis=1) == frame["detection_event_count"]).all()
    assert not (FORBIDDEN & set(frame.columns))
    check_rows.append({
        "split": split_name,
        "rows": len(frame),
        "feature_columns": len(frame.columns),
        "numeric_nonfinite": int((~np.isfinite(numeric)).sum()),
        "basis_categories": ", ".join(sorted(frame["basis"].unique())),
        "missing_feature_values": int(frame.isna().sum().sum()),
        "schema_and_identity_checks": "passed",
    })
checks = pd.DataFrame(check_rows)
display(checks)

In [ ]:
padding_rows = []
for split_name in requested_splits:
    sample = samples[split_name]
    frame = X[split_name]
    total_padding = 0
    nonzero_padding = 0
    for index, row in enumerate(sample):
        steps, width = row["time_steps"], row["width"]
        widths = np.asarray(row["round_widths"], dtype=int)
        grid = np.asarray(row["syndrome"], dtype=int).reshape(steps, width)
        real = np.arange(width)[None, :] < widths[:, None]
        actual_by_step = np.where(real, grid, 0).sum(axis=1)
        actual_count = int(actual_by_step.sum())
        real_slots = int(widths.sum())
        expected_density = actual_count / real_slots if real_slots else 0.0
        assert int(frame["detection_event_count"].iat[index]) == actual_count
        assert np.isclose(frame["event_density"].iat[index], expected_density)
        assert np.array_equal(
            frame.loc[index, list(ACTIVITY_COLUMNS[:steps])].to_numpy(dtype=int), actual_by_step
        )
        assert not frame.loc[index, list(ACTIVITY_COLUMNS[steps:])].to_numpy(dtype=int).any()
        total_padding += int((~real).sum())
        nonzero_padding += int(np.count_nonzero(grid[~real]))
    padding_rows.append({
        "split": split_name, "rows_checked": len(sample),
        "padded_positions": total_padding,
        "nonzero_padded_positions": nonzero_padding,
        "masked_count_density_activity": "passed",
    })
padding_checks = pd.DataFrame(padding_rows)
display(padding_checks)

In [ ]:
# Same hand-calculated cases used by tests/test_features.py. No real shot is printed.
example = {
    "syndrome": [1, 0, 1, 1,  1, 1, 0, 1,  0, 1, 1, 1],
    "round_widths": [2, 3, 1], "time_steps": 3, "width": 4,
    "distance": 3, "rounds": 2, "basis": "X",
}
with_padding = row_features(**example)
assert with_padding["detection_event_count"] == 3
assert with_padding["event_density"] == 3 / 6
assert [with_padding[name] for name in ACTIVITY_COLUMNS[:4]] == [1, 2, 0, 0]
assert with_padding["active_rounds"] == 2
assert with_padding["maximum_round_activity"] == 2
all_zero = row_features(**{**example, "syndrome": [0] * 12})
assert all_zero["detection_event_count"] == 0
assert all_zero["event_density"] == 0.0
zero_real = row_features(**{**example, "round_widths": [0, 0, 0]})
assert zero_real["detection_event_count"] == 0
assert zero_real["event_density"] == 0.0

# Change only forbidden fields on the same synthetic feature row.
base = {name: [value] for name, value in example.items()}
mutated = {**base, "label": [1], "pymatching": [0],
           "correlated_matching": [1], "experiment": ["other"],
           "decoder_correct": [False], "y_router": [1]}
base_X, changed_X = extract_features(base), extract_features(mutated)
assert tuple(base_X) == tuple(changed_X) == expected_columns
assert all(np.array_equal(base_X[name], changed_X[name]) for name in expected_columns)
print("Padded-1, all-zero, zero-real-position, and forbidden-field mutation checks passed.")

## 6. Aggregate feature summaries

Only aggregate split-level statistics appear below. Keep shot-level tables, syndrome arrays, and generated feature files out of Git. These are descriptive checks, not model results.

In [ ]:
summary_features = (
    "detection_event_count", "event_density", "active_rounds",
    "maximum_round_activity", "distance", "rounds",
)
summary_rows = []
for split_name in requested_splits:
    for feature in summary_features:
        values = X[split_name][feature]
        summary_rows.append({
            "split": split_name, "feature": feature,
            "min": values.min(), "median": values.median(),
            "mean": values.mean(), "max": values.max(),
        })
feature_summary = pd.DataFrame(summary_rows)
basis_counts = pd.DataFrame([
    {"split": split_name, "basis": basis, "rows": int((X[split_name]["basis"] == basis).sum())}
    for split_name in requested_splits for basis in VALID_BASES
])
display(feature_summary.round(6))
display(basis_counts)
print("All selected train and validation rows passed feature checks; no test split was requested.")

## 7. Findings handoff

After the Colab run, copy the aggregate settings and tables above into [the Week 2 feature note](../results/tables/week2_feature_checks.md). Record any failed assertion or schema question before Task 3. Notebook 02 should use the same source revision, feature definitions, sample settings, and column order. Do not label these checks as completed until the Colab outputs are saved.